# The EM Algorithm

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/probabilistic-models/02-em-algorithm

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — optimize hidden variables you can't see

**Expectation–Maximization** is the general recipe behind GMMs (and much more): how to fit a model
when some variables are **hidden** (which cluster each point belongs to). It alternates two steps.
The **E-step** computes the expected values of the hidden variables (responsibilities) given the
current parameters; the **M-step** re-estimates the parameters to maximize the expected complete-data
likelihood. The magic property: EM **never decreases** the data log-likelihood — each iteration
provably improves (or holds) it, because EM is maximizing a lower bound (the **ELBO**) that touches
the true likelihood. It converges to a local optimum, and model selection (BIC/AIC) chooses how many
components. We verify the monotonic-improvement guarantee directly.

## EM for 1D GMM

E-step: compute responsibilities $\gamma_{ik} = P(z_i = k | x_i)$

M-step: update $\mu_k, \sigma_k, \pi_k$ using soft assignments

In [ ]:
np.random.seed(42)
n = 300
X = np.concatenate([np.random.randn(150) * 0.8 - 2, np.random.randn(150) * 1.2 + 3])

# Initialize
K = 2
mu = np.array([-3.0, 2.0])
sigma = np.array([1.0, 1.0])
pi = np.array([0.5, 0.5])

log_likelihoods = []
snapshots = []

for iteration in range(30):
    # E-step
    gamma = np.zeros((len(X), K))
    for k in range(K):
        gamma[:, k] = pi[k] * norm.pdf(X, mu[k], sigma[k])
    gamma /= gamma.sum(axis=1, keepdims=True)
    
    # Log-likelihood
    ll = np.sum(np.log(sum(pi[k] * norm.pdf(X, mu[k], sigma[k]) for k in range(K))))
    log_likelihoods.append(ll)
    
    if iteration in [0, 1, 5, 15, 29]:
        snapshots.append((iteration, mu.copy(), sigma.copy(), pi.copy()))
    
    # M-step
    Nk = gamma.sum(axis=0)
    for k in range(K):
        mu[k] = np.sum(gamma[:, k] * X) / Nk[k]
        sigma[k] = np.sqrt(np.sum(gamma[:, k] * (X - mu[k])**2) / Nk[k])
        pi[k] = Nk[k] / len(X)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

x_grid = np.linspace(-6, 8, 300)
for i, (iter_num, mu_s, sigma_s, pi_s) in enumerate(snapshots):
    ax = axes[i]
    ax.hist(X, bins=40, density=True, color='#818cf8', alpha=0.3, edgecolor='#1a1d27')
    mixture = sum(pi_s[k] * norm.pdf(x_grid, mu_s[k], sigma_s[k]) for k in range(K))
    ax.plot(x_grid, mixture, color='#14b8a6', linewidth=2)
    for k in range(K):
        ax.plot(x_grid, pi_s[k] * norm.pdf(x_grid, mu_s[k], sigma_s[k]), '--',
                color=['#f43f5e', '#eab308'][k], linewidth=1)
    ax.set_title(f'Iteration {iter_num}', color='white', fontsize=11)
    ax.set_xlim(-6, 8)

# Log-likelihood plot
ax = axes[5]
ax.plot(log_likelihoods, color='#818cf8', linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Log-Likelihood')
ax.set_title('Monotonic Increase (EM guarantee)', color='white', fontsize=11)

plt.suptitle('EM Algorithm: Iterative Convergence', color='white', fontsize=13)
plt.tight_layout()
plt.show()

**What to notice:** across the six snapshots the two Gaussians slide and reshape until they fit the
data's two bumps — starting from a poor guess and improving every iteration. The E-step (assign soft
responsibilities) and M-step (re-fit the Gaussians) alternate until nothing moves.

## The library way — verify EM's monotonic-improvement guarantee

The defining theorem of EM: the data log-likelihood is **non-decreasing** at every iteration. Our
from-scratch run recorded it, so we can check the guarantee directly.

In [ ]:
ll = np.array(log_likelihoods)
diffs = np.diff(ll)
print('log-likelihood, first few iterations:', ll[:5].round(2))
print('minimum step-to-step change:', diffs.min().round(6), '(must be >= ~0)')
assert np.all(diffs >= -1e-6), "EM must never decrease the log-likelihood"
print('\nEM increased (never decreased) the log-likelihood every iteration ✓')

**What to notice:** every step-to-step change is `≥ 0` — the log-likelihood climbs monotonically to
convergence, exactly as the EM theorem promises. This guarantee is *why* EM is trusted: you never have
to worry about it diverging, only about which local optimum it settles into.

## One EM iteration by hand + the ELBO guarantee

EM is coordinate ascent on the **ELBO**, a Jensen lower bound on $\log p(x\mid\theta)$ whose gap to the true log-likelihood is $\mathrm{KL}(q\,\|\,p(z\mid x,\theta))$. The E-step picks $q=$ posterior (gap $\to 0$, bound tight); the M-step maximizes the bound over $\theta$. Hence $\log p$ never decreases. Below we run **one** iteration on the lesson's 4-point example and confirm the log-likelihood rose from $-7.514$ to $-7.390$.

In [ ]:
import numpy as np
from scipy.stats import norm

Xe = np.array([1.0, 2.0, 4.0, 5.0])
mu, sig, pi = np.array([2.0, 4.0]), np.array([1.5, 1.5]), np.array([0.5, 0.5])

def loglik(mu, sig, pi):
    comp = np.array([pi[k] * norm.pdf(Xe, mu[k], sig[k]) for k in range(2)])  # (K, n)
    return np.log(comp.sum(axis=0)).sum()

print(f'log-likelihood BEFORE = {loglik(mu, sig, pi):.4f}')

# E-step: responsibilities (Bayes)
comp = np.array([pi[k] * norm.pdf(Xe, mu[k], sig[k]) for k in range(2)]).T   # (n, K)
gamma = comp / comp.sum(axis=1, keepdims=True)
print('\nresponsibilities (gamma):')
for xi, g in zip(Xe, gamma):
    print(f'  x={xi:.0f}: {np.round(g, 3)}')

# M-step: responsibility-weighted updates
Nk = gamma.sum(axis=0)
mu_new = (gamma * Xe[:, None]).sum(axis=0) / Nk
sig_new = np.sqrt((gamma * (Xe[:, None] - mu_new)**2).sum(axis=0) / Nk)
pi_new = Nk / len(Xe)
print(f'\nNk={np.round(Nk,3)}  mu_new={np.round(mu_new,3)}  '
      f'sig_new={np.round(sig_new,3)}  pi_new={np.round(pi_new,3)}')

ll_after = loglik(mu_new, sig_new, pi_new)
print(f'\nlog-likelihood AFTER  = {ll_after:.4f}   (rose -> EM guarantee holds: '
      f'{ll_after >= loglik(mu, sig, pi)})')


**What to notice:** the by-hand iteration plus the **ELBO** picture explains *why* the guarantee
holds: EM maximizes a lower bound on the log-likelihood that is **tight** (touches the true likelihood)
after each E-step. Raising the bound in the M-step therefore raises the true likelihood — monotonic
improvement, by construction.

## Model Selection: BIC vs AIC

How many components should we use? Both penalize complexity.

In [ ]:
from sklearn.mixture import GaussianMixture

bic_scores = []
aic_scores = []
for k in range(1, 8):
    gmm = GaussianMixture(n_components=k, random_state=42).fit(X.reshape(-1, 1))
    bic_scores.append(gmm.bic(X.reshape(-1, 1)))
    aic_scores.append(gmm.aic(X.reshape(-1, 1)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 8), bic_scores, 'o-', color='#818cf8', linewidth=2, label='BIC')
ax.plot(range(1, 8), aic_scores, 's-', color='#14b8a6', linewidth=2, label='AIC')
ax.axvline(2, color='#f43f5e', linestyle='--', alpha=0.5, label='True K = 2')
ax.set_xlabel('Number of Components')
ax.set_ylabel('Score (lower is better)')
ax.set_title('BIC vs AIC for Model Selection', color='white')
ax.legend()
plt.tight_layout()
plt.show()

**What to notice:** **BIC** and **AIC** both trade fit against parameter count, but **BIC penalizes
complexity more** (its penalty grows with `log n`), so it favors **fewer** components; AIC is more
permissive and tends to pick **more**. When they disagree, BIC is the conservative choice.

## The log-likelihood increases every iteration

EM is guaranteed to **never decrease** the log-likelihood — it climbs to a local optimum. Watching the curve is the standard convergence check.

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=400, centers=3, cluster_std=1.0, random_state=0)
lls = []
for n in range(1, 21):
    gm = GaussianMixture(n_components=3, max_iter=n, n_init=1,
                         init_params='random', random_state=2).fit(X)
    lls.append(gm.lower_bound_)         # avg log-likelihood at convergence of n iters
plt.plot(range(1, 21), lls, 'o-', color='#14b8a6')
plt.xlabel('max EM iterations'); plt.ylabel('avg log-likelihood')
plt.title('EM monotonically improves the likelihood'); plt.show()

**What to notice:** running EM for more iterations, the average log-likelihood **rises and then
plateaus** — never dipping. This is the same monotonic guarantee visualized on `sklearn`'s solver:
`lower_bound_` (the ELBO) increases each iteration until convergence.

## Gotchas & tradeoffs

- **Monotonic ≠ global.** EM never decreases the likelihood, but it converges to a **local** optimum —
  use multiple restarts.
- **BIC vs AIC can disagree.** AIC (lighter penalty) picks more components; BIC (heavier) fewer. Neither
  is "right" — they optimize different criteria.
- **Singular covariance.** A component collapsing onto a point sends the likelihood to `+∞` — a
  degenerate "improvement" the guarantee doesn't protect against; regularize.
- **Slow near flat regions.** EM can crawl when the likelihood surface is flat; it's simple and stable
  but not always fast.

In [ ]:
# AIC (lighter penalty) tends to select MORE components than BIC (heavier penalty)
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs
Xg, _ = make_blobs(n_samples=400, centers=3, cluster_std=1.6, random_state=0)
aic = [GaussianMixture(k, random_state=0).fit(Xg).aic(Xg) for k in range(1, 9)]
bic = [GaussianMixture(k, random_state=0).fit(Xg).bic(Xg) for k in range(1, 9)]
print(f'AIC picks K = {int(np.argmin(aic)) + 1}   |   BIC picks K = {int(np.argmin(bic)) + 1}')
print('-> AIC is more permissive (more components); BIC is more conservative')

**What to notice:** on the same data AIC and BIC can land on **different `K`** — AIC's lighter penalty
lets it add components BIC rejects. This isn't a bug: they answer different questions (predictive
accuracy vs. finding the "true" model). Report which you used and why.

## Key takeaways

- **EM** alternates an **E-step** (soft responsibilities) and **M-step** (re-estimate parameters).
- It handles **latent variables** — which Gaussian generated each point is unknown.
- The log-likelihood is **non-decreasing**; EM converges to a **local** optimum (use restarts).
- The same E/M pattern powers HMMs (Baum-Welch), missing-data imputation, and more.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The M-step

Given a responsibility matrix $R$ ($n \times K$), the M-step is weighted statistics:

$$N_k = \sum_i r_{ik}, \qquad \pi_k = \frac{N_k}{n}, \qquad \mu_k = \frac{\sum_i r_{ik} x_i}{N_k}, \qquad \sigma_k^2 = \frac{\sum_i r_{ik}(x_i - \mu_k)^2}{N_k}$$

The checks confirm the two limiting cases that make this intuitive: **hard** (0/1) responsibilities reduce to per-cluster means and stds, and **uniform** responsibilities give every component the global statistics.

In [ ]:
def m_step(x, R):
    """Weighted parameter updates. x is (n,), R is (n, K). Returns (pis, mus, sigmas)."""
    x = np.asarray(x, dtype=float)
    R = np.asarray(R, dtype=float)

    # TODO(you): effective counts N_k = column sums of R
    Nk = ...

    # TODO(you): mixing weights N_k / n
    pis = ...

    # TODO(you): weighted means (hint: (R * x[:, None]).sum(axis=0) / Nk)
    mus = ...

    # TODO(you): weighted variances around each mu_k, then sqrt
    var = ...

    return pis, mus, np.sqrt(var)

In [ ]:
# Checks — run me
x = np.array([0.0, 1.0, 10.0, 11.0])

R_hard = np.array([[1, 0], [1, 0], [0, 1], [0, 1]], dtype=float)
pis, mus, sigmas = m_step(x, R_hard)
assert np.allclose(pis, [0.5, 0.5]), "two points each -> equal weights"
assert np.allclose(mus, [0.5, 10.5]), "hard responsibilities reduce to per-cluster means"
assert np.allclose(sigmas, [0.5, 0.5]), "and per-cluster stds"

pis_u, mus_u, sigmas_u = m_step(x, np.full((4, 2), 0.5))
assert np.allclose(mus_u, [x.mean(), x.mean()]), "uniform responsibilities -> every component sees all data"
assert np.allclose(sigmas_u, [x.std(), x.std()]), "and gets the global std"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def m_step(x, R):
    x = np.asarray(x, dtype=float)
    R = np.asarray(R, dtype=float)
    Nk = R.sum(axis=0)
    pis = Nk / len(x)
    mus = (R * x[:, None]).sum(axis=0) / Nk
    var = (R * (x[:, None] - mus[None, :]) ** 2).sum(axis=0) / Nk
    return pis, mus, np.sqrt(var)
```

</details>

### Exercise 2 — BIC and AIC

More components always raise the log-likelihood, so model selection needs a complexity tax:

$$\text{BIC} = k \ln n - 2 \ln \hat{L}, \qquad \text{AIC} = 2k - 2 \ln \hat{L} \qquad (\text{lower is better})$$

Implement both. The checks stage the disagreement from the section above: a +4 log-likelihood gain for 2 extra parameters is **rejected by BIC** at $n = 100$ (penalty $2\ln 100 \approx 9.2$) but **accepted by AIC** (penalty 4) — BIC is the stricter judge.

In [ ]:
def bic(loglik, k_params, n):
    """Bayesian information criterion (lower is better)."""
    # TODO(you): k ln n - 2 LL
    return ...


def aic(loglik, k_params):
    """Akaike information criterion (lower is better)."""
    # TODO(you): 2k - 2 LL
    return ...

In [ ]:
# Checks — run me
assert abs(bic(-100.0, 5, 100) - (5 * np.log(100) + 200)) < 1e-12, "BIC = k ln n - 2 LL"
assert abs(aic(-100.0, 5) - 210.0) < 1e-12, "AIC = 2k - 2 LL"

gain = 4.0   # log-likelihood improvement from 2 extra parameters
assert bic(-100.0 + gain, 7, 100) > bic(-100.0, 5, 100), \
    "BIC: +4 log-likelihood doesn't justify 2 extra params at n=100"
assert aic(-100.0 + gain, 7) < aic(-100.0, 5), "AIC: the same trade is worth it — AIC is more permissive"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def bic(loglik, k_params, n):
    return k_params * np.log(n) - 2 * loglik


def aic(loglik, k_params):
    return 2 * k_params - 2 * loglik
```

</details>